만약 Colab environment인 경우 다음과 같은 코드 실행

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/2026PredictingStructuralStability/src')

Mounted at /content/drive


Local environment의 경우 cd 명령어를 사용하여 working directory 조정

최종적으로, 다음 코드 조각이 이 파일이 속한 디렉토리를 지시해야 함

In [ ]:
print(os.getcwd())

/content/drive/MyDrive/2026PredictingStructuralStability/src


In [ ]:
!pip install -r ../requirements.txt

In [ ]:
import sys
import pandas as pd
import numpy as np
import PIL
import torch
import torchvision
import sklearn
import tqdm
import timm

print("=" * 30)
print("[Info] Library Versions:")
print("=" * 30)
print(f"Python       : {sys.version.split()[0]}")
print(f"PyTorch      : {torch.__version__}")
print(f"Torchvision  : {torchvision.__version__}")
print(f"Pandas       : {pd.__version__}")
print(f"NumPy        : {np.__version__}")
print(f"Pillow (PIL) : {PIL.__version__}")
print(f"Scikit-learn : {sklearn.__version__}")
print(f"tqdm         : {tqdm.__version__}")
print(f"timm         : {timm.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print("=" * 30)

[Info] Library Versions:
Python       : 3.12.13
PyTorch      : 2.10.0+cu128
Torchvision  : 0.25.0+cu128
Pandas       : 2.2.2
NumPy        : 2.0.2
Pillow (PIL) : 11.3.0
Scikit-learn : 1.6.1
tqdm         : 4.67.3
timm         : 1.0.25
CUDA Available: True


In [ ]:
import os
import random
import pandas as pd
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import timm
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm

CFG = {
    # ----------------------------------
    # 1. Base / Training
    # ----------------------------------
    'IMG_SIZE': 384,
    'EPOCHS': 50,
    'LEARNING_RATE': 5e-5,
    'BATCH_SIZE': 16,
    'SEED': 42,
    'PATIENCE': 10,
    'N_FOLDS': 5,

    # ----------------------------------
    # 2. Model & Classifier
    # ----------------------------------
    'MODEL_NAME': 'convnext_tiny',
    'NUM_CLASSES': 1,
    'BACKBONE_OUT': 768,
    'HIDDEN_DIM': 512,
    'DROPOUT': 0.3,

    # ----------------------------------
    # 3. Optimizer & Scheduler
    # ----------------------------------
    'WEIGHT_DECAY': 0.05,
    'WARMUP_EPOCHS': 5,
    'LR_START_FACTOR': 0.1,
    'LR_ETA_MIN': 1e-6,

    # ----------------------------------
    # 4. Data Augmentation & Normalize
    # ----------------------------------
    'AUG_AFFINE_DEGREES': 0,
    'AUG_AFFINE_TRANSLATE': (0.1, 0.1),
    'AUG_CJ_BRIGHTNESS': 0.2,
    'AUG_CJ_CONTRAST': 0.2,
    'AUG_CJ_SATURATION': 0.2,
    'AUG_GRAYSCALE_P': 0.5,
    'AUG_FLIP_P': 0.5,
    'NORM_MEAN': [0.485, 0.456, 0.406],
    'NORM_STD': [0.229, 0.224, 0.225],

    # ----------------------------------
    # 5. Inference & Save
    # ----------------------------------
    'EPS': 1e-15,
    'OUTPUT_PREFIX': '001_convnext_tiny'
}

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(CFG['SEED'])

# ==========================================
# 0. Directory Setup
# ==========================================
base_dir = os.getcwd()
# Dataset directory
data_dir = os.path.abspath(os.path.join(os.path.dirname(base_dir), 'data'))
train_img_dir = os.path.join(data_dir, 'train')
dev_img_dir = os.path.join(data_dir, 'dev')
test_img_dir = os.path.join(data_dir, 'test')

# Model save directory
model_dir = base_dir

# Output directory
output_dir = os.path.abspath(os.path.join(os.path.dirname(base_dir), 'outputs'))
os.makedirs(output_dir, exist_ok=True)


# ==========================================
# 1. Data Loading
# ==========================================
raw_train_df = pd.read_csv(os.path.join(data_dir, 'train.csv'))
raw_dev_df = pd.read_csv(os.path.join(data_dir, 'dev.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'sample_submission.csv'))

# Concatnate train dataset and dev dataset
full_df = pd.concat([raw_train_df, raw_dev_df]).reset_index(drop=True)

# Dataset
class MultiViewDataset(Dataset):
    def __init__(self, df, root_dirs, transform=None, is_test=False):
        self.df = df
        self.root_dirs = root_dirs if isinstance(root_dirs, list) else [root_dirs]
        self.transform = transform
        self.is_test = is_test
        self.label_map = {'stable': 0, 'unstable': 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        sample_id = str(self.df.iloc[idx]['id'])

        actual_folder_path = None
        for root in self.root_dirs:
            potential_path = os.path.join(root, sample_id)
            if os.path.isdir(potential_path):
                actual_folder_path = potential_path
                break

        if actual_folder_path is None:
             raise FileNotFoundError(f"Directory for sample '{sample_id}' not found in any of: {self.root_dirs}")

        views = []
        for name in ["front", "top"]:
            img_path = os.path.join(actual_folder_path, f"{name}.png")
            image = Image.open(img_path).convert("RGB")
            if self.transform:
                image = self.transform(image)
            views.append(image)

        if self.is_test:
            return views

        label = self.label_map[self.df.iloc[idx]['label']]
        return views, label

# Train Augmentation & Preprocess
train_transform = transforms.Compose([
    transforms.RandomAffine(degrees=CFG['AUG_AFFINE_DEGREES'], translate=CFG['AUG_AFFINE_TRANSLATE']),
    transforms.ColorJitter(brightness=CFG['AUG_CJ_BRIGHTNESS'], contrast=CFG['AUG_CJ_CONTRAST'], saturation=CFG['AUG_CJ_SATURATION']),
    transforms.RandomGrayscale(p=CFG['AUG_GRAYSCALE_P']),
    transforms.RandomHorizontalFlip(p=CFG['AUG_FLIP_P']),
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=CFG['NORM_MEAN'], std=CFG['NORM_STD'])
])

# Test Preprocess (No Augmentation)
test_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=CFG['NORM_MEAN'], std=CFG['NORM_STD'])
])


# ==========================================
# 2. Model Definition
# ==========================================
class MultiViewConvNextTinyNet(nn.Module):
    def __init__(self, model_name=CFG['MODEL_NAME'], num_classes=CFG['NUM_CLASSES']):
        super(MultiViewConvNextTinyNet, self).__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)

        self.classifier = nn.Sequential(
            nn.Linear(CFG['BACKBONE_OUT'] * 2, CFG['HIDDEN_DIM']),
            nn.GELU(),
            nn.Dropout(CFG['DROPOUT']),
            nn.Linear(CFG['HIDDEN_DIM'], num_classes)
        )

    def forward(self, views):
        f1 = self.backbone(views[0])
        f2 = self.backbone(views[1])
        combined = torch.cat((f1, f2), dim=1)
        return self.classifier(combined)


# ==========================================
# 3. Training & Validation Functions
# ==========================================
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    train_loss = 0
    for views, labels in tqdm(loader, desc="Training", leave=False):
        views = [v.to(device) for v in views]
        labels = labels.to(device).float()

        optimizer.zero_grad()
        outputs = model(views).view(-1)

        loss = criterion(outputs, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item()
    return train_loss / len(loader)

def validate(model, loader, criterion, device):
    model.eval()
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for views, labels in tqdm(loader, desc="Validation (TTA)", leave=False):
            labels = labels.to(device).float()

            views_orig = [v.to(device) for v in views]
            outputs_orig = model(views_orig).view(-1)
            probs_orig = torch.sigmoid(outputs_orig)

            # TTA with horizontal fliped image
            views_flipped = [torch.flip(v, dims=[3]).to(device) for v in views]
            outputs_flipped = model(views_flipped).view(-1)
            probs_flipped = torch.sigmoid(outputs_flipped)

            probs_tta = (probs_orig + probs_flipped) / 2.0

            all_probs.extend(probs_tta.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_probs = np.array(all_probs, dtype=np.float64)
    all_labels = np.array(all_labels, dtype=np.float64)

    eps = CFG['EPS']
    p = np.clip(all_probs, eps, 1 - eps)
    logloss_score = -np.mean(all_labels * np.log(p) + (1 - all_labels) * np.log(1 - p))
    acc_score = np.mean((all_probs > 0.5) == all_labels)

    return logloss_score, acc_score


# ==========================================
# 4. 5-Fold Cross Validation & Inference
# ==========================================
print("\n--- Starting 5-Fold Cross Validation ---")

# Inference dataset & loader & preds (Shared for each fold iteration)
test_dataset = MultiViewDataset(test_df, test_img_dir, test_transform, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False)
test_preds = np.zeros(len(test_df))

# 5 fold CV
skf = StratifiedKFold(n_splits=CFG['N_FOLDS'], shuffle=True, random_state=CFG['SEED'])

for fold, (train_idx, val_idx) in enumerate(skf.split(full_df, full_df['label'])):
    print(f"\n========== Fold {fold+1}/{CFG['N_FOLDS']} ==========")

    # Split data
    train_fold_df = full_df.iloc[train_idx].reset_index(drop=True)
    val_fold_df = full_df.iloc[val_idx].reset_index(drop=True)

    # Train & val dataset
    train_dataset = MultiViewDataset(train_fold_df, [train_img_dir, dev_img_dir], train_transform, is_test=False)
    val_dataset = MultiViewDataset(val_fold_df, [train_img_dir, dev_img_dir], test_transform, is_test=False)

    # Train & val loader
    train_loader = DataLoader(train_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False)

    # Model & Optimizer & Loss
    model = MultiViewConvNextTinyNet(model_name=CFG['MODEL_NAME']).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=CFG['LEARNING_RATE'], weight_decay=CFG['WEIGHT_DECAY'])

    # Scheduler
    warmup_epochs = CFG['WARMUP_EPOCHS']
    scheduler_warmup = LinearLR(optimizer, start_factor=CFG['LR_START_FACTOR'], total_iters=warmup_epochs)
    scheduler_cosine = CosineAnnealingLR(optimizer, T_max=CFG['EPOCHS'] - warmup_epochs, eta_min=CFG['LR_ETA_MIN'])
    scheduler = SequentialLR(optimizer, schedulers=[scheduler_warmup, scheduler_cosine], milestones=[warmup_epochs])

    # Early Stopping validation
    best_val_logloss = float('inf')
    patience_counter = 0

    # Fold model's path
    best_model_path = os.path.join(model_dir, f"{CFG['OUTPUT_PREFIX']}_fold{fold+1}.pth")

    for epoch in range(1, CFG['EPOCHS'] + 1):
        avg_train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_logloss, val_acc = validate(model, val_loader, criterion, device)

        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step()

        print(f"Epoch [{epoch}/{CFG['EPOCHS']}] | LR: {current_lr:.6f} | Train Loss: {avg_train_loss:.4f} | Val LogLoss: {val_logloss:.4f} | Val Acc: {val_acc:.4f}")

        # Early Stopping
        if val_logloss < best_val_logloss:
            best_val_logloss = val_logloss
            patience_counter = 0
            torch.save(model.state_dict(), best_model_path)
            print(f"-> Saved best model for Fold {fold+1} (LogLoss: {val_logloss:.4f})")
        else:
            patience_counter += 1

        if patience_counter >= CFG['PATIENCE']:
            print(f"Early stopping triggered at epoch {epoch}.")
            break

    # --- Inference for this Fold ---
    print(f"\n--- Inference for Fold {fold+1} ---")
    model.load_state_dict(torch.load(best_model_path))
    model.eval()

    fold_preds = []
    with torch.no_grad():
        for views in tqdm(test_loader, desc=f"Test Preds Fold {fold+1}"):
            views_orig = [v.to(device) for v in views]
            outputs_orig = model(views_orig).view(-1)
            probs_orig = torch.sigmoid(outputs_orig)

            # TTA with horizontal fliped image
            views_flipped = [torch.flip(v, dims=[3]).to(device) for v in views]
            outputs_flipped = model(views_flipped).view(-1)
            probs_flipped = torch.sigmoid(outputs_flipped)

            probs_tta = (probs_orig + probs_flipped) / 2.0
            fold_preds.extend(probs_tta.cpu().numpy())

    # Average 5 fold results (Accumulating Implementation)
    test_preds += np.array(fold_preds) / CFG['N_FOLDS']


# ==========================================
# 5. Final Submission (Ensemble of 5 Folds)
# ==========================================
print("\n--- Generating Final Submission ---")
eps = CFG['EPS']
test_preds = np.clip(test_preds, eps, 1 - eps)

submission = pd.DataFrame({
    'id': test_df['id'],
    'unstable_prob': test_preds,
    'stable_prob': 1.0 - test_preds
})

submission_path = os.path.join(output_dir, f"submission_{CFG['OUTPUT_PREFIX']}.csv")
submission.to_csv(submission_path, encoding='UTF-8-sig', index=False)
print(f"Saved to {submission_path}")


--- Starting 5-Fold Cross Validation ---

========== Fold 1/5 ==========


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

Training:   0%|          | 0/55 [00:00<?, ?it/s]

Validation (TTA):   0%|          | 0/14 [00:00<?, ?it/s]

...

Training:   0%|          | 0/55 [00:00<?, ?it/s]

Validation (TTA):   0%|          | 0/14 [00:00<?, ?it/s]

Epoch [35/50] | LR: 0.000015 | Train Loss: 0.0000 | Val LogLoss: 0.0000 | Val Acc: 1.0000
Early stopping triggered at epoch 35.

--- Inference for Fold 5 ---


Test Preds Fold 5:   0%|          | 0/63 [00:00<?, ?it/s]


--- Generating Final Submission ---
/content/drive/MyDrive/2026PredictingStructuralStability/outputs/001_convnext_tiny.csv 저장 완료.
